# 🏔️ Semana 12 · Unidad 3 — Heapsort

## Información del Curso

| Aspecto | Detalle |
|--------|--------|
| **Universidad** | Universidad de Talca, Chile |
| **Carrera** | Ingeniería Civil en Informática |
| **Semestre** | 2°-3° año |
| **Curso** | Algoritmos y Estructuras de Datos |
| **Docente** | PhD. César Astudillo |
| **Clase** | Semana 12 · Unidad 3 — Heapsort |
| **Duración** | 50 minutos |

---
> 🎯 *Este notebook está diseñado para ser ejecutado en clase de forma interactiva.  
> Ejecuta las celdas en orden de arriba hacia abajo.*

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import random
import time
from IPython.display import display, HTML
print("✅ Dependencias cargadas correctamente")

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Describir** las dos fases de Heapsort: heapify y sortdown.
2. **Implementar** Heapsort in-place usando un arreglo 1-based.
3. **Explicar** por qué Heapsort construye el heap usando sink (no swim) en la fase 1.
4. **Analizar** la complejidad de Heapsort: O(n log n) en el peor caso con O(1) de espacio extra.
5. **Comparar** Heapsort con Mergesort y Quicksort, identificando cuándo usar cada uno.

# Sección 1: Del Heap al Ordenamiento (8 minutos)

## La observación de la clase anterior

Al final de la Clase 6 vimos que si extraemos todos los elementos de un max-heap con `delMax()`, salen **en orden decreciente**.

```python
pq = MaxHeap()
for v in [5, 3, 8, 1, 7, 2, 9, 4, 6]:
    pq.insert(v)

while not pq.isEmpty():
    print(pq.delMax())   # imprime: 9, 8, 7, 6, 5, 4, 3, 2, 1
```

Eso es una ordenación. ¿Podemos hacer esto in-place?

## El problema del enfoque directo

Si usamos la MaxHeap de la clase anterior:
- Necesitamos O(n) de memoria extra para el heap.
- Tenemos que copiar los elementos al heap y luego al arreglo resultado.

**Heapsort resuelve esto** usando el arreglo original como el heap.

## Las dos fases de Heapsort

```
Arreglo desordenado
        ↓
  Fase 1: heapify  →  construir max-heap in-place (el arreglo es el heap)
        ↓
  Fase 2: sortdown →  extraer máximos sucesivos al final del arreglo
        ↓
Arreglo ordenado (ascendente)
```

> 🎙️ **[PAUSA PROFESOR]** *"Si extraemos el máximo y lo ponemos al final, y hacemos eso n veces... ¿dónde va quedando el arreglo ordenado?"*  
> *Respuesta: el arreglo ordenado crece desde el final. Al final, todo el arreglo está ordenado ascendentemente.*

# Sección 2: Fase 1 — Heapify (15 minutos)

## ¿Cómo construir el heap in-place?

**Opción A (ingenua):** Insertar elemento a elemento con swim.  
Costo: n × O(log n) = O(n log n)

**Opción B (eficiente, la de Heapsort):** Aplicar sink de derecha a izquierda, **saltando las hojas**.  
Costo: O(n) — ¡lineal!

## La idea de heapify con sink

Las **hojas** ya satisfacen la propiedad heap (no tienen hijos, no pueden violarla).  
El último nodo interno tiene índice `n // 2`.

```
Arreglo inicial: [_, 4, 1, 3, 2, 9, 7, 8, 5, 6]  (índice 0 sin usar)
                      ↑                ↑
               n//2 = 4               n = 9

Árbol:
         4(1)
        /    \
      1(2)   3(3)
     /  \   /  \
   2(4) 9(5)7(6) 8(7)
   / \
 5(8) 6(9)

Aplicar sink desde posición n//2=4 hasta 1:

sink(4): heap[4]=2, hijos: heap[8]=5, heap[9]=6  → 2 < 6, swap(4,9) → ...
sink(3): heap[3]=3, hijos: heap[6]=7, heap[7]=8  → 3 < 8, swap(3,7) → ...
sink(2): heap[2]=1, hijos: heap[4]=6, heap[5]=9  → 1 < 9, swap(2,5) → ...
sink(1): heap[1]=4, hijos: heap[2]=9, heap[3]=8  → 4 < 9, swap(1,2) → ...
```

## ¿Por qué heapify con sink es O(n) y no O(n log n)?

Las hojas (aprox. n/2 nodos) no hacen ningún intercambio.  
Los nodos en el penúltimo nivel hacen a lo sumo 1 intercambio.  
Solo la raíz puede hacer hasta log₂(n) intercambios.

Suma total: ≈ n/2·0 + n/4·1 + n/8·2 + ... ≈ **2n** intercambios = O(n).

In [ ]:
def sink(arr, k, n):
    """
    Hundir el elemento en posición k dentro de un heap de tamaño n.
    Versión standalone para Heapsort (no necesita la clase MaxHeap).
    Indexación 1-based: arr[1..n]
    """
    while 2 * k <= n:
        j = 2 * k                           # hijo izquierdo
        if j < n and arr[j] < arr[j + 1]:   # elegir el hijo mayor
            j += 1
        if arr[k] >= arr[j]:                # ya en su lugar
            break
        arr[k], arr[j] = arr[j], arr[k]    # swap con el hijo mayor
        k = j


def heapify(arr, n):
    """
    Construir max-heap in-place aplicando sink de derecha a izquierda.
    Solo se procesan los nodos internos (índices n//2 hasta 1).
    """
    for k in range(n // 2, 0, -1):
        sink(arr, k, n)


# Demostración paso a paso
arr = [None, 4, 1, 3, 2, 9, 7, 8, 5, 6]  # índice 0 sin usar
n = len(arr) - 1

print(f"Arreglo inicial:     {arr[1:]}")
print(f"Nodos internos: posiciones 1 a {n // 2} (posiciones {n//2+1} a {n} son hojas)")
print()

for k in range(n // 2, 0, -1):
    antes = arr[1:].copy()
    sink(arr, k, n)
    print(f"  sink({k}): {antes} → {arr[1:]}")

print(f"\nMax-heap construido: {arr[1:]}")
print(f"Máximo en arr[1] = {arr[1]}  ✓")

In [ ]:
# Comparar costo de heapify con sink vs inserción secuencial con swim
def heapify_swim(arr, n):
    """Construir heap insertando uno a uno (costoso)."""
    comparaciones = 0
    def swim_count(arr, k):
        nonlocal comparaciones
        while k > 1:
            comparaciones += 1
            if arr[k // 2] < arr[k]:
                arr[k // 2], arr[k] = arr[k], arr[k // 2]
                k = k // 2
            else:
                break
    for i in range(2, n + 1):
        swim_count(arr, i)
    return comparaciones

def heapify_sink_count(arr, n):
    """Construir heap con sink desde n//2 (eficiente)."""
    comparaciones = 0
    def sink_count(arr, k, n):
        nonlocal comparaciones
        while 2 * k <= n:
            j = 2 * k
            if j < n:
                comparaciones += 1
                if arr[j] < arr[j + 1]: j += 1
            comparaciones += 1
            if arr[k] >= arr[j]: break
            arr[k], arr[j] = arr[j], arr[k]
            k = j
    for k in range(n // 2, 0, -1):
        sink_count(arr, k, n)
    return comparaciones

ns = [100, 500, 1000, 5000, 10000, 50000]
comp_swim_list = []
comp_sink_list = []

for n in ns:
    datos = random.sample(range(n * 10), n)
    arr1 = [None] + datos[:]
    arr2 = [None] + datos[:]
    comp_swim_list.append(heapify_swim(arr1, n))
    comp_sink_list.append(heapify_sink_count(arr2, n))

plt.figure(figsize=(10, 5))
plt.plot(ns, comp_swim_list, 'ro-', label='heapify con swim — O(n log n)', linewidth=2)
plt.plot(ns, comp_sink_list, 'bo-', label='heapify con sink — O(n)', linewidth=2)
plt.plot(ns, ns, 'g--', label='n (referencia lineal)', linewidth=1.5)
plt.xlabel('n')
plt.ylabel('Comparaciones')
plt.title('Heapify: swim vs sink')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"{'n':>8} {'Swim (n log n)':>16} {'Sink (n)':>12} {'Ratio':>8}")
print("-" * 48)
for i, n in enumerate(ns):
    ratio = comp_swim_list[i] / comp_sink_list[i]
    print(f"{n:>8,} {comp_swim_list[i]:>16,} {comp_sink_list[i]:>12,} {ratio:>8.1f}x")

# Sección 3: Fase 2 — Sortdown (12 minutos)

## La idea de sortdown

Una vez que el arreglo es un max-heap, extraemos el máximo **n veces**.

**Truco in-place:** En vez de guardar el máximo en un arreglo separado, lo ponemos en la última posición libre del arreglo original:

```
Estado inicial (heap válido, n=9):
[_, 9, 6, 8, 5, 2, 7, 3, 4, 1]
    ↑máx                    ↑última

Paso 1: swap(1, 9), n=8, sink(1)
[_, 8, 6, 7, 5, 2, 1, 3, 4, |9|]
                             └─ ordenado

Paso 2: swap(1, 8), n=7, sink(1)
[_, 7, 6, 3, 5, 2, 1, 4, |8, 9|]
                          └─ ordenado
...

Paso n: el arreglo completo está ordenado ascendentemente
[_, 1, 2, 3, 4, 5, 6, 7, 8, 9]
```

> 📌 Clave: el máximo extraído **ocupa la posición que acaba de quedar libre** al final del heap. El heap se encoge y el arreglo ordenado crece desde el final.

In [ ]:
def heapsort(arr):
    """
    Heapsort in-place.
    Trabaja sobre una copia interna con indexación 1-based.
    
    Fase 1: heapify con sink — O(n)
    Fase 2: sortdown         — O(n log n)
    Total:                     O(n log n)
    Espacio extra:             O(1)
    """
    n = len(arr)
    # Convertir a 1-based internamente (arreglo de trabajo)
    h = [None] + arr[:]  # h[0] sin usar

    # ── Fase 1: heapify ──────────────────────────────
    for k in range(n // 2, 0, -1):
        sink(h, k, n)

    # ── Fase 2: sortdown ─────────────────────────────
    while n > 1:
        h[1], h[n] = h[n], h[1]   # mover máximo al final
        n -= 1                     # encoger el heap
        sink(h, 1, n)              # restaurar heap-order

    # Copiar de vuelta (0-based)
    for i in range(len(arr)):
        arr[i] = h[i + 1]


# Demostración con tracing
def heapsort_verbose(arr_original):
    arr = arr_original[:]
    n_orig = len(arr)
    h = [None] + arr
    n = n_orig

    print(f"Arreglo inicial:    {h[1:]}")
    print()
    print("── Fase 1: Heapify ──")
    for k in range(n // 2, 0, -1):
        antes = h[1:].copy()
        sink(h, k, n)
        if antes != h[1:]:
            print(f"  sink({k}): {antes} → {h[1:]}")
    print(f"  Max-heap: {h[1:]}")
    print()
    print("── Fase 2: Sortdown ──")
    paso = 1
    while n > 1:
        h[1], h[n] = h[n], h[1]
        n -= 1
        sink(h, 1, n)
        heap_part    = h[1:n+1]
        sorted_part  = h[n+1:]
        print(f"  Paso {paso:2d}: heap={heap_part}  ordenado={sorted_part}")
        paso += 1

    print(f"\nArreglo ordenado:   {h[1:]}")


heapsort_verbose([4, 1, 3, 2, 9, 7, 8, 5, 6])

# Sección 4: Análisis de Complejidad (8 minutos)

## Complejidad de Heapsort

| Fase | Operaciones | Complejidad |
|------|------------|-------------|
| Heapify (fase 1) | n/2 llamadas a sink | **O(n)** |
| Sortdown (fase 2) | n extracciones × sink | **O(n log n)** |
| **Total** | | **O(n log n)** |

## La propiedad única de Heapsort

Heapsort es el **único algoritmo de comparación** que cumple las tres condiciones simultáneamente:

| Propiedad | Heapsort | Mergesort | Quicksort |
|-----------|:--------:|:---------:|:---------:|
| O(n log n) peor caso | ✅ | ✅ | ❌ (O(n²)) |
| O(1) espacio extra | ✅ | ❌ (O(n)) | ✅ |
| Estable | ❌ | ✅ | ❌ |

**Introsort** (usado en C++ `std::sort`) combina Quicksort + Heapsort: usa Quicksort normalmente pero cambia a Heapsort si detecta recursión demasiado profunda (señal del peor caso de Quicksort).

## ¿Por qué Heapsort no se usa más en la práctica?

A pesar de sus garantías teóricas, Heapsort suele ser más lento que Quicksort en la práctica por:

1. **Cache-unfriendly:** Los accesos al heap saltan por posiciones `k`, `2k`, `2k+1` — muy poco localidad de caché.
2. **Muchas comparaciones:** sink compara dos hijos en cada nivel (2 comparaciones por nivel vs 1 en Quicksort).
3. **No aprovecha patrones:** No tiene ventaja en datos parcialmente ordenados (Timsort sí).

> 🎙️ **[PAUSA PROFESOR]** *"Heapsort tiene la mejor garantía teórica: O(n log n) siempre + O(1) espacio. ¿Por qué Python no lo usa?"*  
> *Respuestas esperadas: cache locality, estabilidad, Timsort aprovecha runs.*

In [ ]:
# Verificar que Heapsort funciona correctamente
casos_prueba = [
    ([5, 3, 8, 1, 7, 2, 9, 4, 6], "Aleatorio"),
    ([1, 2, 3, 4, 5, 6, 7, 8, 9], "Ya ordenado"),
    ([9, 8, 7, 6, 5, 4, 3, 2, 1], "Inversamente ordenado"),
    ([3, 1, 4, 1, 5, 9, 2, 6, 5, 3], "Con repetidos"),
    ([42], "Un elemento"),
    ([], "Vacío"),
]

print("Verificación de Heapsort:")
print("-" * 55)
for datos, nombre in casos_prueba:
    arr = datos[:]
    heapsort(arr)
    correcto = arr == sorted(datos)
    estado = "✅" if correcto else "❌"
    print(f"  {estado} {nombre:<28} → {arr}")

# Benchmark Heapsort vs Quicksort vs sorted() para n grande
import sys
sys.setrecursionlimit(50000)

def quicksort_bench(arr, lo=0, hi=None):
    if hi is None: hi = len(arr) - 1
    if lo < hi:
        pivot = arr[hi]
        i = lo - 1
        for j in range(lo, hi):
            if arr[j] <= pivot:
                i += 1
                arr[i], arr[j] = arr[j], arr[i]
        arr[i+1], arr[hi] = arr[hi], arr[i+1]
        p = i + 1
        quicksort_bench(arr, lo, p - 1)
        quicksort_bench(arr, p + 1, hi)

def mergesort_bench(arr):
    if len(arr) <= 1: return arr
    mid = len(arr) // 2
    izq = mergesort_bench(arr[:mid])
    der = mergesort_bench(arr[mid:])
    resultado, i, j = [], 0, 0
    while i < len(izq) and j < len(der):
        if izq[i] <= der[j]: resultado.append(izq[i]); i += 1
        else:                 resultado.append(der[j]); j += 1
    return resultado + izq[i:] + der[j:]

print("\nBenchmark de tiempo (n=5000, datos aleatorios):")
print("-" * 50)

n_bench = 5000
datos_base = random.sample(range(n_bench * 10), n_bench)

resultados_tiempo = {}
for nombre, fn, necesita_copia in [
    ('Heapsort',  lambda a: heapsort(a),       True),
    ('Quicksort', lambda a: quicksort_bench(a), True),
    ('Mergesort', lambda a: mergesort_bench(a), False),
    ('sorted()',  lambda a: sorted(a),          False),
]:
    tiempos = []
    for _ in range(5):
        arr = datos_base[:]
        t0 = time.perf_counter()
        fn(arr)
        tiempos.append(time.perf_counter() - t0)
    promedio = sum(tiempos) / len(tiempos) * 1000
    resultados_tiempo[nombre] = promedio
    print(f"  {nombre:<12} {promedio:>8.2f} ms")

print()
base = resultados_tiempo['sorted()']
for nombre, ms in resultados_tiempo.items():
    print(f"  {nombre:<12} {ms/base:>5.1f}× más lento que sorted()")

# Sección 5: El Gran Cuadro — Comparando los 3 Algoritmos O(n log n) (7 minutos)

In [ ]:
# Visualización comparativa: comportamiento en distintos tipos de input
tipos_input = {
    'Aleatorio':       lambda n: random.sample(range(n*10), n),
    'Ya ordenado':     lambda n: list(range(n)),
    'Casi ordenado':   lambda n: list(range(n-5)) + random.sample(range(n-5, n*2), 5),
    'Inversamente':    lambda n: list(range(n, 0, -1)),
}

ns = [500, 1000, 2000, 3000, 5000]

def medir_tiempo(fn, datos):
    arr = datos[:]
    t0 = time.perf_counter()
    fn(arr)
    return (time.perf_counter() - t0) * 1000

algoritmos = {
    'Heapsort':  lambda a: heapsort(a),
    'Quicksort': lambda a: quicksort_bench(a),
    'sorted()':  lambda a: sorted(a),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colores = {'Heapsort': '#e74c3c', 'Quicksort': '#3498db', 'sorted()': '#2ecc71'}

for idx, (tipo, gen) in enumerate(tipos_input.items()):
    ax = axes[idx // 2][idx % 2]
    for nombre, fn in algoritmos.items():
        tiempos = []
        for n in ns:
            datos = gen(n)
            t = medir_tiempo(fn, datos)
            tiempos.append(t)
        ax.plot(ns, tiempos, 'o-', color=colores[nombre], label=nombre, linewidth=2)
    ax.set_title(f'Input: {tipo}', fontsize=11, fontweight='bold')
    ax.set_xlabel('n')
    ax.set_ylabel('Tiempo (ms)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Comparación de algoritmos O(n log n) por tipo de input', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Sección 6: Resumen de la Unidad 3 — Ordenamiento (5 minutos)

## Los 6 algoritmos que estudiamos

| Algoritmo | Mejor caso | Caso promedio | Peor caso | Espacio | Estable |
|-----------|:----------:|:------------:|:---------:|:-------:|:-------:|
| Selection Sort | O(n²) | O(n²) | O(n²) | O(1) | ❌ |
| Insertion Sort | O(n) | O(n²) | O(n²) | O(1) | ✅ |
| Shell Sort | O(n log n) | O(n^1.3) | O(n²) | O(1) | ❌ |
| Merge Sort | O(n log n) | O(n log n) | O(n log n) | O(n) | ✅ |
| Quicksort | O(n log n) | O(n log n) | O(n²) | O(log n) | ❌ |
| **Heapsort** | **O(n log n)** | **O(n log n)** | **O(n log n)** | **O(1)** | **❌** |

## ¿Cuándo usar cada uno?

- **¿Necesitas estabilidad?** → Merge Sort o Timsort (`sorted()` de Python)
- **¿RAM limitada y datos aleatorios?** → Quicksort con mediana de 3
- **¿Garantía O(n log n) en peor caso + O(1) espacio?** → Heapsort (o Introsort)
- **¿Datos casi ordenados?** → Insertion Sort o Timsort
- **¿N pequeño (< 50)?** → Insertion Sort directamente

## Conexión con la Unidad 4

En la Unidad 4 (Diccionarios / Búsqueda) veremos estructuras que van más allá del ordenamiento:

- **BST** (Binary Search Tree): O(log n) en búsqueda, inserción y eliminación
- **Red-Black BST**: BST autobalanceado, O(log n) garantizado
- **Hash Tables**: O(1) amortizado, pero sin ordenamiento

La Priority Queue (heap) reaparecerá en gráfos: **Dijkstra** y **Prim** la usan como estructura central.

---

## Referencias

- Sedgewick & Wayne. *Algorithms*, 4ª ed., Sección 2.4
- Williams, J.W.J. (1964). "Algorithm 232: Heapsort". *Communications of the ACM*, 7(6), 347–348.
- Floyd, R.W. (1964). "Algorithm 245: Treesort 3". *Communications of the ACM*, 7(12), 701.
- Cormen et al. *Introduction to Algorithms* (CLRS), Cap. 6

## 📚 Lecturas Recomendadas y Práctica

### Textbooks

| Libro | Edición | Capítulo | Tema |
|-------|---------|----------|------|
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 6.4 | Heapsort |
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 6.3 | Por qué heapify es O(n) |
| Goodrich, Tamassia & Goldwasser (GTG) — *Data Structures and Algorithms in Python* | 1ª ed. | Cap. 9.4 | Heapsort in-place |
| Bhargava (Grok) — *Grokking Algorithms* | 2ª ed. | Cap. 2, 4 | Comparación de ordenamientos |

### Recursos gratuitos en línea

- 🌐 [VisuAlgo — Binary Heap](https://visualgo.net/en/heap) — heapify y sortdown.
- 🎬 [Sorting Algorithms Visualized](https://www.toptal.com/developers/sorting-algorithms) — los seis algoritmos de la unidad, lado a lado.

### Práctica en Codeforces (soporta Python 3)

> 🔍 **Cómo filtrar:** ve a [codeforces.com/problemset](https://codeforces.com/problemset),
> escribe la etiqueta en **Tags** y ajusta **Rating**.

**Escala de dificultad orientativa para este curso:**

| Rating | Nivel | Descripción |
|--------|-------|-------------|
| 800 | ⭐ | Aplicación directa — la mayoría puede resolverlo |
| 1000–1200 | ⭐⭐ | Requiere una pequeña adaptación |
| 1300+ | ⭐⭐⭐ | Combina la idea con otra — desafío |

**Problemas recomendados para este tópico:**

| # | Problema | Rating | Por qué es útil |
|---|----------|--------|-----------------|
| 1 | [1092B — Teams Forming](https://codeforces.com/problemset/problem/1092/B) | ⭐ 800 | Ordenar y emparejar |
| 2 | [1526C1 — Potions (Easy Version)](https://codeforces.com/problemset/problem/1526/C1) | ⭐⭐ 1500 | Heap en su uso natural: mantener el mínimo de lo elegido |
| 3 | [1077C — Good Array](https://codeforces.com/problemset/problem/1077/C) | ⭐⭐⭐ 1300 | Razonar sobre extremos tras ordenar |

⚠️ Los dos primeros son el **mínimo esperado**. Los demás son desafío opcional.

## 🧪 Ejercicio 1: Heapsort descendente ⭐

**Descripción:** Heapsort con un **max**-heap deja el arreglo en orden ascendente. Escribe
una versión que lo deje en orden **descendente**.

Hay dos caminos: invertir el resultado al final, o usar un **min**-heap. Este ejercicio
pide el segundo, porque obliga a entender qué hace realmente `sink`.

**Entrada:** `arr` (list).
**Salida:** el mismo arreglo ordenado de mayor a menor, **in-place**.

**Ejemplo:**
```
Entrada: [3, 1, 4, 1, 5]
Salida:  [5, 4, 3, 1, 1]
```

**Restricciones:** no uses `sorted()`, `list.sort()` ni `reverse()`. Ordena in-place.
**Complejidad esperada:** O(n log n) temporal, O(1) espacial

> 💡 **Pista:** copia `sink` cambiando el sentido de las comparaciones para que hunda hacia
> el **menor**. Con un min-heap, sacar el mínimo repetidamente y ponerlo al final produce
> orden descendente.

In [ ]:
def heapsort_descendente(arr):
    """
    Ordena el arreglo de mayor a menor, in-place, usando un MIN-heap.

    Parámetros:
        arr (list): arreglo a ordenar
    Retorna:
        list: el mismo arreglo, ya ordenado descendentemente
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_ejercicio_1(fn):
    """Verifica el orden descendente y que sea in-place."""
    import random, time
    casos = [
        ([3,1,4,1,5], "el ejemplo del enunciado"),
        ([], "arreglo vacío"),
        ([7], "un elemento"),
        ([1,2,3,4,5], "ya ascendente"),
        ([5,4,3,2,1], "ya descendente"),
        ([2,2,2,2], "todos iguales"),
        ([-5,3,-1,0,9,-2], "con negativos"),
        ([random.randint(0,100) for _ in range(200)], "aleatorio n=200"),
    ]
    aprobados=0
    for arr, desc in casos:
        esperado=sorted(arr, reverse=True)
        a=list(arr)
        t0=time.perf_counter()
        try:
            ret=fn(a); t1=time.perf_counter()
            inplace = ret is None or ret is a
            if a==esperado and inplace:
                print(f"  ✅ {desc} ({(t1-t0)*1000:.2f}ms)"); aprobados+=1
            elif not inplace:
                print(f"  ❌ {desc} — debe ordenar IN-PLACE (retornar el mismo objeto o None)")
            else:
                print(f"  ❌ {desc}\n     Esperado: {esperado[:10]}\n     Obtenido: {a[:10]}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados==len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_ejercicio_1(heapsort_descendente)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def heapsort_descendente(arr):
#     """Min-heap + sortdown: el mínimo va al final, así queda descendente."""
#     def sink_min(a, i, n):
#         """Hunde a[i] hacia el MENOR: es sink con las comparaciones invertidas."""
#         while 2*i + 1 < n:
#             j = 2*i + 1
#             if j + 1 < n and a[j+1] < a[j]:
#                 j += 1                     # nos quedamos con el hijo MENOR
#             if a[i] <= a[j]:
#                 break
#             a[i], a[j] = a[j], a[i]
#             i = j
#     n = len(arr)
#     # Paso 1: construir el min-heap, O(n)
#     for i in range(n//2 - 1, -1, -1):
#         sink_min(arr, i, n)
#     # Paso 2: sacar el mínimo y ponerlo al final, encogiendo el heap
#     for m in range(n, 1, -1):
#         arr[0], arr[m-1] = arr[m-1], arr[0]
#         sink_min(arr, 0, m-1)
#     return arr
#     # Complejidad: O(n log n) en todos los casos, O(1) espacial

## 🧪 Ejercicio 2: Ordenar un arreglo casi ordenado ⭐⭐

**Descripción:** un arreglo está **k-ordenado** si cada elemento está a lo más a `k`
posiciones de su lugar definitivo. Ordénalo en $O(n\log k)$ en vez de $O(n\log n)$.

Es un caso real: registros que llegan casi en orden por *timestamp*, con pequeños desfases
de red.

**Entrada:** `arr` (list) y `k` (int).
**Salida:** una lista nueva, ordenada ascendentemente.

**Ejemplo:**
```
Entrada: arr = [6, 5, 3, 2, 8, 10, 9], k = 3
Salida:  [2, 3, 5, 6, 8, 9, 10]
```

**Restricciones:** no uses `sorted()` sobre el arreglo completo. Puedes usar `heapq`.
**Complejidad esperada:** O(n log k)

> 💡 **Pista:** mantén un min-heap con `k+1` elementos. El mínimo de esa ventana es
> necesariamente el siguiente elemento del resultado: nada más lejos puede ser menor.

In [ ]:
def ordenar_k_ordenado(arr, k):
    """
    Ordena un arreglo k-ordenado en O(n log k) con un min-heap de tamaño k+1.

    Parámetros:
        arr (list): arreglo casi ordenado
        k (int): desplazamiento máximo de cada elemento
    Retorna:
        list: lista nueva, ordenada ascendentemente
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_ejercicio_2(fn):
    """Genera arreglos k-ordenados reales y compara contra sorted()."""
    import random, time
    casos = [
        (([6,5,3,2,8,10,9], 3), "el ejemplo del enunciado"),
        (([], 3), "arreglo vacío"),
        (([1], 0), "un elemento, k=0"),
        (([1,2,3,4], 0), "ya ordenado, k=0"),
        (([2,1,4,3,6,5], 1), "pares intercambiados, k=1"),
        (([3,3,3,3], 2), "todos iguales"),
    ]
    aprobados=0
    for (arr,k), desc in casos:
        esperado=sorted(arr)
        t0=time.perf_counter()
        try:
            r=fn(list(arr),k); t1=time.perf_counter()
            if list(r)==esperado:
                print(f"  ✅ {desc} ({(t1-t0)*1000:.2f}ms) -> {list(r)[:8]}"); aprobados+=1
            else:
                print(f"  ❌ {desc}\n     Esperado: {esperado}\n     Obtenido: {list(r)}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    # arreglos k-ordenados generados de verdad: ordenar y desplazar a lo más k
    random.seed(9); ok=True
    try:
        for _ in range(80):
            n=random.randint(1,60); k=random.randint(0,5)
            a=sorted(random.randint(0,100) for _ in range(n))
            # Para que el arreglo sea REALMENTE k-ordenado, barajamos dentro de
            # bloques disjuntos de tamaño k+1: así ningún elemento se aleja más
            # de k posiciones de su lugar definitivo. Intercambios sucesivos no
            # sirven, porque se componen y pueden mover un elemento mucho más lejos.
            for ini in range(0, n, k+1):
                bloque=a[ini:ini+k+1]
                random.shuffle(bloque)
                a[ini:ini+k+1]=bloque
            if list(fn(list(a),k))!=sorted(a): ok=False; break
        print(f"  {'✅' if ok else '❌'} 80 arreglos k-ordenados generados aleatoriamente")
        aprobados+=ok
    except Exception as e:
        print(f"  💥 pruebas aleatorias — Error: {e}")
    total=len(casos)+1
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados==total else f'⚠️  {aprobados}/{total} casos correctos'}")

verificar_ejercicio_2(ordenar_k_ordenado)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def ordenar_k_ordenado(arr, k):
#     """Ventana deslizante de k+1 elementos en un min-heap."""
#     import heapq
#     if not arr:
#         return []
#     # Paso 1: cargar los primeros k+1 elementos
#     h = arr[:k+1]
#     heapq.heapify(h)                      # O(k)
#     salida = []
#     # Paso 2: por cada elemento restante, sacar el mínimo y meter el nuevo
#     for x in arr[k+1:]:
#         salida.append(heapq.heappushpop(h, x))    # O(log k)
#     # Paso 3: vaciar lo que quedó en el heap
#     while h:
#         salida.append(heapq.heappop(h))
#     return salida
#     # Complejidad: O(n log k) temporal, O(k) espacial
#     # Con k = n-1 degenera a O(n log n), que es heapsort